# depends on DestinationStates

In [1]:
import pandas as pd
from multicall import Call
import numpy as np
from web3 import Web3


from mainnet_launch.database.schema.full import (
    DestinationTokenValues,
    TokenValues,
    Autopools,
    DestinationStates,
    DestinationTokens,
    Destinations,
    AutopoolDestinationStates,
    Tokens,
)
import plotly.express as px


from mainnet_launch.abis import STATS_CALCULATOR_REGISTRY_ABI
from mainnet_launch.data_fetching.get_events import fetch_events


from mainnet_launch.database.schema.postgres_operations import (
    get_full_table_as_orm,
    get_full_table_as_df,
    insert_avoid_conflicts,
    get_subset_not_already_in_column,
    natural_left_right_using_where,
)
from mainnet_launch.data_fetching.get_state_by_block import (
    get_raw_state_by_blocks,
    safe_normalize_with_bool_success,
    build_blocks_to_use,
    identity_with_bool_success,
    get_state_by_one_block,
)
from mainnet_launch.constants import (
    ALL_CHAINS,
    ROOT_PRICE_ORACLE,
    ChainData,
    STATS_CALCULATOR_REGISTRY,
    WETH,
    ETH_CHAIN,
)

from mainnet_launch.pages.autopool_diagnostics.lens_contract import (
    fetch_autopool_to_active_destinations_over_this_period_of_missing_blocks,
)


import pandas as pd
from multicall import Call
import numpy as np
from web3 import Web3


from mainnet_launch.database.schema.full import (
    DestinationStates,
    DestinationTokenValues,
    AutopoolDestinationStates,
    Autopools,
    DestinationTokens,
    Destinations,
    Tokens,
)
from mainnet_launch.database.schema.postgres_operations import (
    get_full_table_as_df,
    get_full_table_as_orm,
    insert_avoid_conflicts,
    get_highest_value_in_field_where,
    get_subset_not_already_in_column,
)
from mainnet_launch.data_fetching.get_state_by_block import (
    get_raw_state_by_blocks,
    safe_normalize_with_bool_success,
    build_blocks_to_use,
    get_state_by_one_block,
)
from mainnet_launch.pages.autopool_diagnostics.lens_contract import (
    fetch_active_destinations_by_autopool_by_block,
    fetch_pools_and_destinations_df,
)
from mainnet_launch.constants import (
    AutopoolConstants,
    ALL_AUTOPOOLS,
    AUTO_LRT,
    POINTS_HOOK,
    ChainData,
)


chain = ETH_CHAIN

possible_blocks = build_blocks_to_use(chain)

missing_blocks = get_subset_not_already_in_column(
    AutopoolDestinationStates,
    AutopoolDestinationStates.block,
    possible_blocks,
    where_clause=AutopoolDestinationStates.chain_id == chain.chain_id,
)

autopool_df = get_full_table_as_df(Autopools, where_clause=Autopools.chain_id == chain.chain_id)
full_destination_df = natural_left_right_using_where(
    DestinationTokens,
    Destinations,
    using=[DestinationTokens.destination_vault_address, DestinationTokens.chain_id],
    where_clause=DestinationTokens.chain_id == chain.chain_id,
)

token_value_df = natural_left_right_using_where(
    DestinationTokenValues,
    TokenValues,
    using=[DestinationTokenValues.block, DestinationTokens.chain_id, DestinationTokens.token_address],
    where_clause=DestinationTokenValues.chain_id == chain.chain_id,
)
token_df = get_full_table_as_df(Tokens, where_clause=Tokens.chain_id == chain.chain_id)
token_value_df = pd.merge(
    token_value_df, token_df[["symbol", "decimals", "token_address"]], how="left", on="token_address"
)
token_value_df

2025-04-27 11:16:59,795 INFO sqlalchemy.engine.Engine select pg_catalog.version()
2025-04-27 11:16:59,798 INFO sqlalchemy.engine.Engine [raw sql] {}
2025-04-27 11:16:59,979 INFO sqlalchemy.engine.Engine select current_schema()
2025-04-27 11:16:59,979 INFO sqlalchemy.engine.Engine [raw sql] {}
2025-04-27 11:17:00,162 INFO sqlalchemy.engine.Engine show standard_conforming_strings
2025-04-27 11:17:00,164 INFO sqlalchemy.engine.Engine [raw sql] {}
2025-04-27 11:17:00,346 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-04-27 11:17:00,365 INFO sqlalchemy.engine.Engine SELECT max(blocks.block) AS block 
FROM blocks 
WHERE blocks.chain_id = %(chain_id_1)s AND blocks.block >= %(block_1)s AND blocks.block <= %(block_2)s GROUP BY date_trunc(%(date_trunc_1)s, blocks.datetime) ORDER BY date_trunc(%(date_trunc_2)s, blocks.datetime)
2025-04-27 11:17:00,366 INFO sqlalchemy.engine.Engine [generated in 0.00062s] {'chain_id_1': 1, 'block_1': 20752910, 'block_2': 100000000, 'date_trunc_1': 'day', 'dat

,block,chain_id,token_address,destination_vault_address,spot_price,quantity,safe_spot_spread,spot_backing_discount,denomiated_in,backing,safe_price,safe_backing_discount,symbol,decimals
0,20759464,1,0x856c4Efb76C1D1AE02e20CEB03A2A6a08b0b8dC3,0x1C6560857400A1C2CDb28B171d1Ee2B882f73E4D,0.999578,4.888206e+03,0.000381,-0.094527,0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2,1.103929,0.999197,-0.094872,OETH,18
1,20759464,1,0xCd5fE23C85820F7B72D0926FC9b05b43E359b7ee,0x148Ca723BefeA7b021C399413b8b7426A4701500,1.047578,7.163209e+02,-0.000142,-0.000422,0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2,1.048020,1.047726,-0.000280,weETH,18
2,20759464,1,0xCd5fE23C85820F7B72D0926FC9b05b43E359b7ee,0x40219bBda953ca811d2D0168Dc806a96b84791d9,1.047578,NaN,-0.000142,-0.000422,0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2,1.048020,1.047726,-0.000280,weETH,18
3,20759464,1,0xCd5fE23C85820F7B72D0926FC9b05b43E359b7ee,0x5c6aeb9ef0d5BbA4E6691f381003503FD0D45126,1.047657,NaN,-0.000066,-0.000346,0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2,1.048020,1.047726,-0.000280,weETH,18
4,20759464,1,0xCd5fE23C85820F7B72D0926FC9b05b43E359b7ee,0x777FAf85c8E5FC6f4332E56B989C5C94201A273C,1.047657,9.067885e+02,-0.000066,-0.000346,0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2,1.048020,1.047726,-0.000280,weETH,18
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35835,22356658,1,0xBe9895146f7AF43049ca1c1AE358B0541Ea49704,0x92294A62D6D9F0FbE30Ba3B543edb1806561baD7,1.095270,2.391526e-02,-0.000201,-0.003355,0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2,1.098958,1.095490,-0.003155,cbETH,18
35836,22356658,1,0xBe9895146f7AF43049ca1c1AE358B0541Ea49704,0xe3AE2Ab9AE8ADe1B4940dd893C9339401bEe61A1,1.095270,2.391526e-02,-0.000201,-0.003355,0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2,1.098958,1.095490,-0.003155,cbETH,18
35837,22356658,1,0x4c9EDD5852cd905f086C759E8383e09bff1E68B3,0xd7900d87069C815a299bdA7aFDcd7eEe98fe4b6c,0.000549,4.646141e+06,-0.001027,NaN,0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2,NaN,0.000549,NaN,USDe,18
35838,22356658,1,0xA663B02CF0a4b149d2aD41910CB81e23e1c41c32,0x4686CEe2B3b1Da28a1db926443d5DDd674462b29,0.000618,0.000000e+00,-0.000840,NaN,0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2,NaN,0.000619,NaN,sFRAX,18


In [ ]:
def build_autopool_balance_of_calls_by_destination(
    autopool_vault_address: str, destination_vault_addresses: list[str]
) -> list[Call]:
    return [
        Call(
            destination_vault_address,
            ["balanceOf(address)(uint256)", autopool_vault_address],
            [((autopool_vault_address, destination_vault_address, "balanceOf"), safe_normalize_with_bool_success)],
        )
        for destination_vault_address in destination_vault_addresses
    ]


def build_destinations_underlyingTotalSupply_calls(destination_vault_addresses: list[str]) -> list[Call]:
    return [
        Call(
            destination_vault_address,
            ["underlyingTotalSupply()(uint256)"],
            [((destination_vault_address, "underlyingTotalSupply"), safe_normalize_with_bool_success)],
        )
        for destination_vault_address in destination_vault_addresses
    ]


def fetch_destination_total_supply_df(
    autopool_to_all_ever_active_destinations: dict, missing_blocks: list[int], chain: ChainData
) -> pd.DataFrame:
    all_active_destinations = set()

    for autopool_vault_address in autopool_to_all_ever_active_destinations.keys():
        this_autopool_active_destinations = [
            dest.destination_vault_address for dest in autopool_to_all_ever_active_destinations[autopool_vault_address]
        ]
        all_active_destinations.update(this_autopool_active_destinations)

    calls = build_destinations_underlyingTotalSupply_calls(list(all_active_destinations))
    destination_total_supply_df = get_raw_state_by_blocks(calls, missing_blocks, chain, include_block_number=True)
    return destination_total_supply_df


def fetch_autopool_balance_of_by_destination(
    autopool_to_all_ever_active_destinations: dict[str, list[Destinations]], missing_blocks: list[int], chain: ChainData
) -> pd.DataFrame:
    autopool_balance_of_calls = []

    for autopool_vault_address in autopool_to_all_ever_active_destinations.keys():
        this_autopool_active_destinations = [
            dest.destination_vault_address for dest in autopool_to_all_ever_active_destinations[autopool_vault_address]
        ]

        autopool_balance_of_calls.extend(
            build_autopool_balance_of_calls_by_destination(autopool_vault_address, this_autopool_active_destinations)
        )

    autopool_destination_balance_of_df = get_raw_state_by_blocks(
        autopool_balance_of_calls, missing_blocks, chain, include_block_number=True
    )
    return autopool_destination_balance_of_df


def _build_destination_points_calls(this_autopool_active_destinations: list[str], chain: ChainData) -> list[Call]:
    return [
        Call(
            POINTS_HOOK(chain),
            ["destinationBoosts(address)(uint256)", destination_vault_address],
            [((destination_vault_address, "points"), safe_normalize_with_bool_success)],
        )
        for destination_vault_address in this_autopool_active_destinations
    ]


def fetch_autopool_points_apr(
    autopool_to_all_ever_active_destinations: dict[str, list[Destinations]], missing_blocks: list[int], chain: ChainData
) -> pd.DataFrame:
    autopool_points_calls = []

    for autopool_vault_address in autopool_to_all_ever_active_destinations.keys():
        this_autopool_active_destinations = [
            dest.destination_vault_address for dest in autopool_to_all_ever_active_destinations[autopool_vault_address]
        ]

        autopool_points_calls.extend(_build_destination_points_calls(this_autopool_active_destinations, chain))

    autopool_points_df = get_raw_state_by_blocks(
        autopool_points_calls, missing_blocks, chain, include_block_number=True
    )
    return autopool_points_df


def _clean_summary_stats_info(success, summary_stats):
    if success is True:
        summary = {
            "destination": summary_stats[0],  # address
            "baseApr": summary_stats[1] / 1e18,
            "feeApr": summary_stats[2] / 1e18,
            "incentiveApr": summary_stats[3] / 1e18,
            "safeTotalSupply": summary_stats[4] / 1e18,
            "priceReturn": summary_stats[5] / 1e18,
            "maxDiscount": summary_stats[6] / 1e18,
            "maxPremium": summary_stats[7] / 1e18,
            "ownedShares": summary_stats[8] / 1e18,
            "compositeReturn": summary_stats[9] / 1e18,
            "pricePerShare": summary_stats[10] / 1e18,
            "pointsApr": None,  # set later
        }
        return summary
    else:
        return None


def _build_summary_stats_call(
    autopool: Autopools,
    dest: Destinations,
    direction: str = "out",
    amount: int = 0,
) -> Call:
    # /// @notice Gets the safe price of the underlying LP token
    # /// @dev Price validated to be inside our tolerance against spot price. Will revert if outside.
    # /// @return price Value of 1 unit of the underlying LP token in terms of the base asset
    # function getValidatedSafePrice() external returns (uint256 price);
    # getDestinationSummaryStats uses getValidatedSafePrice. So when prices are outside tolerance this function reverts

    # consider finding a version of this function that won't revert, (follow up, I am pretty sure that does not exist)
    if direction == "in":
        direction_enum = 0
    elif direction == "out":
        direction_enum = 1
    return_types = "(address,uint256,uint256,uint256,uint256,int256,int256,int256,uint256,int256,uint256)"

    # cleaning_function = build_summary_stats_cleaning_function(autopool)
    return Call(
        autopool.strategy_address,
        [
            f"getDestinationSummaryStats(address,uint8,uint256)({return_types})",
            dest.destination_vault_address,
            direction_enum,
            amount,
        ],
        [((autopool.autopool_vault_address, dest.destination_vault_address, direction), _clean_summary_stats_info)],
    )


def fetch_destination_summary_stats_df(
    autopool_to_all_ever_active_destinations: dict, missing_blocks: list[int], chain: ChainData
) -> pd.DataFrame:
    autopools_orm: list[Autopools] = get_full_table_as_orm(Autopools, where_clause=Autopools.chain_id == chain.chain_id)
    full_autopool_summary_stats_df = None

    for autopool in autopools_orm:
        all_summary_stats_calls = []
        this_autopool_destinations = autopool_to_all_ever_active_destinations[autopool.autopool_vault_address]
        for dest in this_autopool_destinations:
            all_summary_stats_calls.append(_build_summary_stats_call(autopool, dest, "out"))
            all_summary_stats_calls.append(_build_summary_stats_call(autopool, dest, "in"))

        autopool_summary_stats_df = get_raw_state_by_blocks(
            all_summary_stats_calls, missing_blocks, chain, include_block_number=True
        )

        if full_autopool_summary_stats_df is None:
            full_autopool_summary_stats_df = autopool_summary_stats_df.copy()
        else:
            full_autopool_summary_stats_df = pd.merge(
                full_autopool_summary_stats_df, autopool_summary_stats_df, on="block"
            )

    return full_autopool_summary_stats_df


def _extract_new_destination_states(
    autopool_summary_stats_df: pd.DataFrame,
    destination_underlying_total_supply_df: pd.DataFrame,
    autopool_points_df: pd.DataFrame,
    token_value_df: pd.DataFrame,
    autopool_to_all_ever_active_destinations: dict[str | list[Destinations]],
):

    all_new_destination_states = []
    # autopool_summary_stats_df, destination_underlying_total_supply_df, token_value_df, autopool_to_all_ever_active_destinations
    raw_destination_states_df = pd.merge(autopool_summary_stats_df, destination_underlying_total_supply_df, on="block")
    raw_destination_states_df = pd.merge(raw_destination_states_df, autopool_points_df, on="block")

    for autopool_vault_address in autopool_to_all_ever_active_destinations.keys():
        for dest in autopool_to_all_ever_active_destinations[autopool_vault_address]:
            just_this_destination_df = token_value_df[
                token_value_df["destination_vault_address"] == dest.destination_vault_address
            ].copy()

            # total value in the destination,
            just_this_destination_df["total_spot_value"] = (
                just_this_destination_df["quantity"] * just_this_destination_df["spot_price"]
            )
            just_this_destination_df["total_safe_value"] = (
                just_this_destination_df["quantity"] * just_this_destination_df["safe_price"]
            )
            just_this_destination_df["total_backing_value"] = (
                just_this_destination_df["quantity"] * just_this_destination_df["backing"]
            )

            this_destination_total_value = (
                just_this_destination_df.groupby("block")[
                    ["total_spot_value", "total_safe_value", "total_backing_value"]
                ]
                .sum()
                .reset_index()
            )

            local_df = pd.merge(this_destination_total_value, raw_destination_states_df, on=["block"])

            def _extract_destination_states(row: pd.DataFrame) -> None:
                # try to pull the "in" and "out" stats, defaulting to an empty dict
                in_summary_stats = row.get((autopool_vault_address, dest.destination_vault_address, "in"), {}) or {}
                out_summary_stats = row.get((autopool_vault_address, dest.destination_vault_address, "out"), {}) or {}

                total_apr_in = in_summary_stats.get("compositeReturn")
                total_apr_out = out_summary_stats.get("compositeReturn")

                incentive_apr = in_summary_stats.get("incentiveApr")
                fee_apr = in_summary_stats.get("feeApr")
                base_apr = in_summary_stats.get("baseApr")

                price_per_share = in_summary_stats.get("pricePerShare")
                price_return = in_summary_stats.get("priceReturn")
                fee_plus_base_apr = None  # only for post autoUSD destinations
                safe_total_supply = in_summary_stats.get("safeTotalSupply")

                points_apr = row[(dest.destination_vault_address, "points")]
                underlying_total_supply = row[(dest.destination_vault_address, "underlyingTotalSupply")]

                # the value of a lp token
                underlying_safe_price = row["total_safe_value"] / underlying_total_supply
                underlying_spot_price = row["total_spot_value"] / underlying_total_supply
                underlying_backing = row["total_backing_value"] / underlying_total_supply

                safe_backing_discount = (underlying_safe_price - underlying_backing) / underlying_backing
                safe_spot_spread = (underlying_safe_price - underlying_spot_price) / underlying_safe_price
                spot_backing_discount = (underlying_spot_price - underlying_backing) / underlying_backing

                new_destination_state = DestinationStates(
                    destination_vault_address=dest.destination_vault_address,
                    block=row["block"],
                    chain_id=chain.chain_id,
                    incentive_apr=incentive_apr,
                    fee_apr=fee_apr,
                    base_apr=base_apr,
                    points_apr=points_apr,
                    fee_plus_base_apr=fee_plus_base_apr,
                    total_apr_in=total_apr_in,
                    total_apr_out=total_apr_out,
                    underlying_token_total_supply=underlying_total_supply,
                    safe_total_supply=safe_total_supply,
                    price_per_share=price_per_share,
                    price_return=price_return,
                    underlying_safe_price=underlying_safe_price,
                    underlying_spot_price=underlying_spot_price,
                    underlying_backing=underlying_backing,
                    safe_backing_discount=safe_backing_discount,
                    spot_backing_discount=spot_backing_discount,
                    safe_spot_spread=safe_spot_spread,
                )
                all_new_destination_states.append(new_destination_state)

            local_df.apply(_extract_destination_states, axis=1)

    return all_new_destination_states


def ensure_destination_states_is_current():
    for chain in ALL_CHAINS:
        possible_blocks = build_blocks_to_use(chain)

        missing_blocks = get_subset_not_already_in_column(
            AutopoolDestinationStates,
            AutopoolDestinationStates.block,
            possible_blocks,
            where_clause=AutopoolDestinationStates.chain_id == chain.chain_id,
        )

        autopool_df = get_full_table_as_df(Autopools, where_clause=Autopools.chain_id == chain.chain_id)

        full_destination_df = natural_left_right_using_where(
            DestinationTokens,
            Destinations,
            using=[DestinationTokens.destination_vault_address, DestinationTokens.chain_id],
            where_clause=DestinationTokens.chain_id == chain.chain_id,
        )

        token_value_df = natural_left_right_using_where(
            DestinationTokenValues,
            TokenValues,
            using=[DestinationTokenValues.block, DestinationTokens.chain_id, DestinationTokens.token_address],
            where_clause=DestinationTokenValues.chain_id == chain.chain_id,
        )
        # not sure here on the way to specify only a subset of oclumsn? maybe with type hints? add later, eg columsn = none and
        token_df = get_full_table_as_df(Tokens, where_clause=Tokens.chain_id == chain.chain_id)[
            ["symbol", "decimals", "token_address"]
        ]
        token_value_df = pd.merge(token_value_df, token_df, how="left", on="token_address")

        autopool_to_all_ever_active_destinations = (
            fetch_autopool_to_active_destinations_over_this_period_of_missing_blocks(chain, missing_blocks)
        )

        destination_underlying_total_supply_df = fetch_destination_total_supply_df(
            autopool_to_all_ever_active_destinations, missing_blocks, chain
        )

        # autopool_destination_balance_of_df = fetch_autopool_balance_of_by_destination(
        #     autopool_to_all_ever_active_destinations, missing_blocks, chain
        # )  # only needed for autopool destination state, the rest can be extracted from the other tables

        autopool_points_df = fetch_autopool_points_apr(autopool_to_all_ever_active_destinations, missing_blocks, chain)

        autopool_summary_stats_df = fetch_destination_summary_stats_df(
            autopool_to_all_ever_active_destinations, missing_blocks, chain
        )

        all_new_destination_states = _extract_new_destination_states(
            autopool_summary_stats_df,
            destination_underlying_total_supply_df,
            autopool_points_df,
            token_value_df,
            autopool_to_all_ever_active_destinations,
        )
        insert_avoid_conflicts(
            all_new_destination_states,
            DestinationStates,
            index_elements=[
                DestinationStates.block,
                DestinationStates.chain_id,
                DestinationStates.destination_vault_address,
            ],
        )

2025-04-27 11:17:03,364 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-04-27 11:17:03,365 INFO sqlalchemy.engine.Engine SELECT pg_catalog.pg_class.relname 
FROM pg_catalog.pg_class JOIN pg_catalog.pg_namespace ON pg_catalog.pg_namespace.oid = pg_catalog.pg_class.relnamespace 
WHERE pg_catalog.pg_class.relname = %(table_name)s AND pg_catalog.pg_class.relkind = ANY (ARRAY[%(param_1)s, %(param_2)s, %(param_3)s, %(param_4)s, %(param_5)s]) AND pg_catalog.pg_table_is_visible(pg_catalog.pg_class.oid) AND pg_catalog.pg_namespace.nspname != %(nspname_1)s
2025-04-27 11:17:03,365 INFO sqlalchemy.engine.Engine [cached since 2.437s ago] {'table_name': <sqlalchemy.sql.elements.TextClause object at 0x11aa7aa40>, 'param_1': 'r', 'param_2': 'p', 'param_3': 'f', 'param_4': 'v', 'param_5': 'm', 'nspname_1': 'pg_catalog'}
2025-04-27 11:17:03,365 INFO sqlalchemy.engine.Engine 
            SELECT *
            FROM autopools
            WHERE autopools.chain_id = 1
        
2025-04-27 11:17:03,366 INFO

[Errno 1] [SSL: SSLV3_ALERT_BAD_RECORD_MAC] ssl/tls alert bad record mac (_ssl.c:2578) [0]
[Errno 1] [SSL: SSLV3_ALERT_BAD_RECORD_MAC] ssl/tls alert bad record mac (_ssl.c:2578) [0]
[Errno 1] [SSL: SSLV3_ALERT_BAD_RECORD_MAC] ssl/tls alert bad record mac (_ssl.c:2578) [0]
[Errno 1] [SSL: SSLV3_ALERT_BAD_RECORD_MAC] ssl/tls alert bad record mac (_ssl.c:2578) [0]
[Errno 1] [SSL: SSLV3_ALERT_BAD_RECORD_MAC] ssl/tls alert bad record mac (_ssl.c:2578) [0]
[Errno 1] [SSL: SSLV3_ALERT_BAD_RECORD_MAC] ssl/tls alert bad record mac (_ssl.c:2578) [0]


2025-04-27 11:17:35,567 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-04-27 11:17:35,568 INFO sqlalchemy.engine.Engine 
            SELECT *
            FROM autopools
            WHERE autopools.chain_id = 1
        
2025-04-27 11:17:35,568 INFO sqlalchemy.engine.Engine [cached since 34.64s ago] {}
2025-04-27 11:17:35,752 INFO sqlalchemy.engine.Engine COMMIT


[Errno 1] [SSL: SSLV3_ALERT_BAD_RECORD_MAC] ssl/tls alert bad record mac (_ssl.c:2578) [0]
[Errno 1] [SSL: SSLV3_ALERT_BAD_RECORD_MAC] ssl/tls alert bad record mac (_ssl.c:2578) [0]
[Errno 1] [SSL: SSLV3_ALERT_BAD_RECORD_MAC] ssl/tls alert bad record mac (_ssl.c:2578) [0]
[Errno 1] [SSL: SSLV3_ALERT_BAD_RECORD_MAC] ssl/tls alert bad record mac (_ssl.c:2578) [0]


In [ ]:
def _extract_new_destination_states(
    autopool_summary_stats_df: pd.DataFrame,
    destination_underlying_total_supply_df: pd.DataFrame,
    autopool_points_df: pd.DataFrame,
    token_value_df: pd.DataFrame,
):

    all_new_destination_states = []
    # autopool_summary_stats_df, destination_underlying_total_supply_df, token_value_df, autopool_to_all_ever_active_destinations
    raw_destination_states_df = pd.merge(autopool_summary_stats_df, destination_underlying_total_supply_df, on="block")
    raw_destination_states_df = pd.merge(raw_destination_states_df, autopool_points_df, on="block")

    for autopool_vault_address in autopool_to_all_ever_active_destinations.keys():
        for dest in autopool_to_all_ever_active_destinations[autopool_vault_address]:

            destination_tokens = full_destination_df[
                full_destination_df["destination_vault_address"] == dest.destination_vault_address
            ]["token_address"]

            just_this_destination_df = token_value_df[
                token_value_df["destination_vault_address"] == dest.destination_vault_address
            ].copy()

            # total value in the destination,
            just_this_destination_df["total_spot_value"] = (
                just_this_destination_df["quantity"] * just_this_destination_df["spot_price"]
            )
            just_this_destination_df["total_safe_value"] = (
                just_this_destination_df["quantity"] * just_this_destination_df["safe_price"]
            )
            just_this_destination_df["total_backing_value"] = (
                just_this_destination_df["quantity"] * just_this_destination_df["backing"]
            )

            this_destination_total_value = (
                just_this_destination_df.groupby("block")[
                    ["total_spot_value", "total_safe_value", "total_backing_value"]
                ]
                .sum()
                .reset_index()
            )

            local_df = pd.merge(this_destination_total_value, raw_destination_states_df, on=["block"])

            def _extract_destination_states(row: pd.DataFrame) -> None:
                # try to pull the "in" and "out" stats, defaulting to an empty dict
                in_summary_stats = row.get((autopool_vault_address, dest.destination_vault_address, "in"), {}) or {}
                out_summary_stats = row.get((autopool_vault_address, dest.destination_vault_address, "out"), {}) or {}

                total_apr_in = in_summary_stats.get("compositeReturn")
                total_apr_out = out_summary_stats.get("compositeReturn")

                incentive_apr = in_summary_stats.get("incentiveApr")
                fee_apr = in_summary_stats.get("feeApr")
                base_apr = in_summary_stats.get("baseApr")

                price_per_share = in_summary_stats.get("pricePerShare")
                price_return = in_summary_stats.get("priceReturn")
                fee_plus_base_apr = None  # only for post autoUSD destinations
                safe_total_supply = in_summary_stats.get("safeTotalSupply")

                points_apr = row[(dest.destination_vault_address, "points")]
                underlying_total_supply = row[(dest.destination_vault_address, "underlyingTotalSupply")]

                # the value of a lp token
                underlying_safe_price = row["total_safe_value"] / underlying_total_supply
                underlying_spot_price = row["total_spot_value"] / underlying_total_supply
                underlying_backing = row["total_backing_value"] / underlying_total_supply

                safe_backing_discount = (underlying_safe_price - underlying_backing) / underlying_backing
                safe_spot_spread = (underlying_safe_price - underlying_spot_price) / underlying_safe_price
                spot_backing_discount = (underlying_spot_price - underlying_backing) / underlying_backing

                new_destination_state = DestinationStates(
                    destination_vault_address=dest.destination_vault_address,
                    block=row["block"],
                    chain_id=chain.chain_id,
                    incentive_apr=incentive_apr,
                    fee_apr=fee_apr,
                    base_apr=base_apr,
                    points_apr=points_apr,
                    fee_plus_base_apr=fee_plus_base_apr,
                    total_apr_in=total_apr_in,
                    total_apr_out=total_apr_out,
                    underlying_token_total_supply=underlying_total_supply,
                    safe_total_supply=safe_total_supply,
                    price_per_share=price_per_share,
                    price_return=price_return,
                    underlying_safe_price=underlying_safe_price,
                    underlying_spot_price=underlying_spot_price,
                    underlying_backing=underlying_backing,
                    safe_backing_discount=safe_backing_discount,
                    spot_backing_discount=spot_backing_discount,
                    safe_spot_spread=safe_spot_spread,
                )
                all_new_destination_states.append(new_destination_state)

            local_df.apply(_extract_destination_states, axis=1)

    return all_new_destination_states

2025-04-27 11:18:09,591 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-04-27 11:18:26,532 INFO sqlalchemy.engine.Engine COMMIT


In [8]:
df = get_full_table_as_df(DestinationStates)
df

2025-04-27 11:19:05,197 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-04-27 11:19:05,198 INFO sqlalchemy.engine.Engine SELECT pg_catalog.pg_class.relname 
FROM pg_catalog.pg_class JOIN pg_catalog.pg_namespace ON pg_catalog.pg_namespace.oid = pg_catalog.pg_class.relnamespace 
WHERE pg_catalog.pg_class.relname = %(table_name)s AND pg_catalog.pg_class.relkind = ANY (ARRAY[%(param_1)s, %(param_2)s, %(param_3)s, %(param_4)s, %(param_5)s]) AND pg_catalog.pg_table_is_visible(pg_catalog.pg_class.oid) AND pg_catalog.pg_namespace.nspname != %(nspname_1)s
2025-04-27 11:19:05,198 INFO sqlalchemy.engine.Engine [cached since 124.3s ago] {'table_name': <sqlalchemy.sql.elements.TextClause object at 0x11d28a950>, 'param_1': 'r', 'param_2': 'p', 'param_3': 'f', 'param_4': 'v', 'param_5': 'm', 'nspname_1': 'pg_catalog'}
2025-04-27 11:19:05,198 INFO sqlalchemy.engine.Engine 
            SELECT *
            FROM destination_states
            
        
2025-04-27 11:19:05,199 INFO sqlalchemy.engine.

,destination_vault_address,block,chain_id,incentive_apr,fee_apr,base_apr,points_apr,fee_plus_base_apr,total_apr_in,total_apr_out,underlying_token_total_supply,safe_total_supply,price_per_share,price_return,underlying_safe_price,underlying_spot_price,underlying_backing,safe_backing_discount,spot_backing_discount,safe_spot_spread
0,0xf3ae3c74EaD129e770A864CeE291A805b170bBe0,20759464,1,0.041235,0.003997,0.005817,0.0,None,0.046281,0.046281,13157.451200,12302.611124,1.036523,-0.000645,1.036523,1.036516,1.035855,0.000645,0.000638,6.621342e-06
1,0xf3ae3c74EaD129e770A864CeE291A805b170bBe0,20766617,1,0.040938,0.003618,0.005802,0.0,None,0.045987,0.045987,13157.496413,12383.288934,1.036163,-0.000278,1.036163,1.036121,1.035875,0.000278,0.000237,4.066212e-05
2,0xf3ae3c74EaD129e770A864CeE291A805b170bBe0,20773761,1,0.040507,0.003295,0.005848,0.0,None,0.045061,0.045061,13157.496413,12383.288934,1.036462,-0.000538,1.036462,1.036456,1.035906,0.000537,0.000531,6.483657e-06
3,0xf3ae3c74EaD129e770A864CeE291A805b170bBe0,20780916,1,0.040196,0.003022,0.005817,0.0,None,0.044440,0.044440,13268.458946,12479.834854,1.036491,-0.000575,1.036491,1.036477,1.035896,0.000574,0.000560,1.429726e-05
4,0xf3ae3c74EaD129e770A864CeE291A805b170bBe0,20788084,1,0.040448,0.002794,0.005885,0.0,None,0.044524,0.044524,13268.526874,12494.174709,1.036501,-0.000558,1.036501,1.036502,1.035924,0.000557,0.000558,-6.380341e-07
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8507,0x777FAf85c8E5FC6f4332E56B989C5C94201A273C,22327989,1,0.000000,0.000000,0.000000,0.0,None,0.000000,0.000000,5048.902326,0.000000,1.045577,0.000000,1.045577,1.045772,1.046107,-0.000506,-0.000320,-1.863948e-04
8508,0x777FAf85c8E5FC6f4332E56B989C5C94201A273C,22335153,1,0.000000,0.000000,0.000000,0.0,None,0.000000,0.000000,5048.902326,0.000000,1.045815,0.000000,1.045815,1.045912,1.046169,-0.000338,-0.000246,-9.195140e-05
8509,0x777FAf85c8E5FC6f4332E56B989C5C94201A273C,22342307,1,0.000000,0.000000,0.000000,0.0,None,0.000000,0.000000,5117.872032,0.000000,1.045629,0.000000,1.045629,1.045839,1.046240,-0.000584,-0.000383,-2.012162e-04
8510,0x777FAf85c8E5FC6f4332E56B989C5C94201A273C,22349480,1,0.000000,0.000000,0.000000,0.0,None,0.000000,0.000000,4975.260393,0.000000,1.045822,0.000000,1.045822,1.045994,1.046303,-0.000459,-0.000295,-1.649598e-04


In [4]:
just_this_destination_df[["backing", "symbol"]]

,backing,symbol
95,1.0,pxETH
151,1.0,WETH
255,1.0,pxETH
311,1.0,WETH
417,1.0,pxETH
...,...,...
35511,1.0,WETH
35615,1.0,pxETH
35671,1.0,WETH
35775,1.0,pxETH


In [5]:
# maybe we get the apply map version?

In [6]:
# class DestinationStates(Base):
#     __tablename__ = "destination_states"

#     destination_vault_address: Mapped[str] = mapped_column(primary_key=True)
#     block: Mapped[int] = mapped_column(primary_key=True)
#     chain_id: Mapped[int] = mapped_column(primary_key=True)
#     # information about the destination itself at this moment in time

#     incentive_apr: Mapped[float] = mapped_column(nullable=False)
#     fee_and_base_apr: Mapped[float] = mapped_column(nullable=False)
#     points_apr: Mapped[float] = mapped_column(nullable=True)

#     total_apr_in: Mapped[float] = mapped_column(
#         nullable=True
#     )  # get destination summaryStats (in, and out) are seperate calls
#     total_apr_out: Mapped[float] = mapped_column(nullable=True)

#     underlying_token_total_staked: Mapped[float] = mapped_column(nullable=True)
#     underlying_token_total_supply: Mapped[float] = mapped_column(nullable=False)
#     safe_total_supply: Mapped[float] = mapped_column(nullable=True)  # only for pre autoUSD destinations

#     # this is as lp tokens # via
#     underlying_safe_price: Mapped[float] = mapped_column(nullable=False)
#     underlying_spot_price: Mapped[float] = mapped_column(nullable=False)
#     underlying_backing: Mapped[float] = mapped_column(nullable=False)
#     denominated_in: Mapped[str] = mapped_column(nullable=False) # should live in the destination

#     safe_backing_discount: Mapped[float] = mapped_column(nullable=True)
#     safe_spot_spread: Mapped[float] = mapped_column(nullable=True)
#     spot_backing_discount: Mapped[float] = mapped_column(nullable=True)

#     __table_args__ = (
#         ForeignKeyConstraint(["block", "chain_id"], ["blocks.block", "blocks.chain_id"]),
#         ForeignKeyConstraint(
#             ["destination_vault_address", "chain_id"],
#             ["destinations.destination_vault_address", "destinations.chain_id"],
#         ),
#     )

In [7]:
# drop this
AutopoolDestinationStates_new_partial_rows = []

def _to_apply_over_each_row(row:dict):
    
    for autopool_dest_tuple, balance_of in row.items():
        if autopool_dest_tuple != 'block':
            autopool_vault_address, destination_vault_address = col # unpack the tuple

            AutopoolDestinationStates_new_partial_rows.append(
                        {
            'autopool_vault_address':autopool_vault_address,
            'destination_vault_address':destination_vault_address, 
            'block': row['block'],
            'amount': balance_of,
            # this will raise an error if trying to insert into AutopoolDestinationStates because nullable = false 
            'total_safe_value': None,
            'total_spot_value': None,
            'total_backing_value': None,
            'percent_ownership': -100.0, # depends on destination_states.underlying_token_total_supply
        }

            )

# t



for col in autopool_destination_balance_of_df.columns:
    this_autopool_destination_balance_of = []
    autopool_balance_columns = autopool_destination_balance_of_df[col]
    autopool_vault_address, destination_vault_address = col # unpack
    this_autopool_destination_balance_of.append(
        {
            'autopool_vault_address':autopool_vault_address,
            'destination_vault_address':

        }

    )



SyntaxError: expression expected after dictionary key and ':' (2675838443.py, line 36)

In [ ]:
autopool_to_all_ever_active_destinations.keys()

In [ ]:
autopool_to_all_ever_active_destinations.keys

In [ ]:
# def build_pool_token_spot_price_calls(
#     chain: ChainData, pool_addresses: list[str], token_addresses: list[str]
# ) -> list[Call]:

#     return [
#         Call(
#             ROOT_PRICE_ORACLE(chain),
#             ["getSpotPriceInEth(address,address)(uint256)", token_address, pool_address],
#             [(f"{pool_address}_{token_address}", safe_normalize_with_bool_success)],
#         )
#         for (pool_address, token_address) in zip(pool_addresses, token_addresses)
#     ]


# spot_price_calls = build_pool_token_spot_price_calls(
#     chain, full_destination_df["pool"], full_destination_df["token_address"]
# )
# destination_token_spot_price_df = get_raw_state_by_blocks(
#     spot_price_calls, possible_blocks, chain, include_block_number=True
# )
# destination_token_spot_price_df


# def build_underlying_reserves_calls(destinations: list[str]) -> list[Call]:
#     return [
#         Call(
#             dest,
#             "underlyingReserves()(address[],uint256[])",
#             [
#                 (f"{dest}_underlyingReserves_tokens", identity_with_bool_success),
#                 (f"{dest}_underlyingReserves_amounts", identity_with_bool_success),
#             ],
#         )
#         for dest in destinations
#     ]


# underlying_reserves_calls = build_underlying_reserves_calls(full_destination_df["destination_vault_address"])
# underlying_reserves_df = get_raw_state_by_blocks(
#     underlying_reserves_calls, possible_blocks, chain, include_block_number=True
# )
# underlying_reserves_df